# ДЗ 5 — QLoRA-дообучение и профайлинг

Задачи:
1. Датасет для генерации текста
2. Предобученная модель (0.5B–1.5B)
3. Оценка до обучения: «корзинка» примеров + lm-evaluation-harness
4. QLoRA-дообучение
5. Оценка после обучения
6. Профайлинг через `torch.profiler`

Ноутбук рассчитан на Google Colab с GPU (T4 хватает). Запускать ячейки по порядку.

## 0. Установка зависимостей

Версии зафиксированы — это и часть воспроизводимости, и страховка от ломающих изменений в `transformers`/`peft`/`bitsandbytes`.

In [1]:
!pip -q install -U \
    "transformers==4.46.3" \
    "datasets==3.1.0" \
    "accelerate==1.1.1" \
    "peft==0.13.2" \
    "bitsandbytes>=0.45.0" \
    "trl==0.12.1" \
    "lm-eval==0.4.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00


In [2]:
import os, random, json, gc, time
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Включаем детерминизм по возможности; полный determinism с QLoRA не гарантирован,
# но фиксации seed + cudnn benchmark=False достаточно для воспроизводимости общего поведения.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

device: cuda
Tesla T4


## 1. Датасет

Беру `IlyaGusev/ru_turbo_alpaca` — русскоязычный instruction-following корпус (формат `instruction / input / output`, сгенерирован в стиле Self-Instruct поверх ChatGPT). Это именно то, под что в реальной жизни обычно дообучают на русском: разнообразные инструкции с понятным форматом.

 Берём срез 4000 train / 200 eval — баланс между видимостью эффекта и временем прогона.

In [3]:
from datasets import load_dataset

# trust_remote_code=True — у датасета есть свой loader-скрипт
raw = load_dataset("IlyaGusev/ru_turbo_alpaca", split="train", trust_remote_code=True)
raw = raw.shuffle(seed=SEED)

# Срезы делаем устойчивыми к фактическому размеру датасета
N_EVAL  = 200
N_TRAIN = min(4000, max(0, len(raw) - N_EVAL))
train_ds = raw.select(range(N_TRAIN))
eval_ds  = raw.select(range(N_TRAIN, N_TRAIN + N_EVAL))

print(f"train: {len(train_ds)}, eval: {len(eval_ds)}")
print("колонки:", train_ds.column_names)
print("\nПример записи:\n", train_ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train: 4000, eval: 200
колонки: ['instruction', 'input', 'output', 'alternative_output', 'label', 'all_labels', 'agreement', 'overlap']

Пример записи:
 {'instruction': 'Найди пропущенные слова и вставь их в правильном порядке.', 'input': 'Я понял, что ___ жалею о том, что ___  начал ___ этот класс раньше.', 'output': 'Я понял, что я жалею о том, что не начал посещать этот класс раньше.', 'alternative_output': 'Я понял, что я жалею о том, что не начал я этот класс раньше.', 'label': 'ok', 'all_labels': ['ok', 'ok', 'ok'], 'agreement': 1.0, 'overlap': 3}


In [4]:
# Русскоязычный prompt-шаблон в стиле Alpaca. Без специальных chat-токенов —
# чтобы получить чистое сравнение «база vs QLoRA», без вмешательства chat-template.
PROMPT_WITH_INPUT = (
    "Ниже приведена инструкция, описывающая задачу, и входные данные с дополнительным контекстом. "
    "Напиши ответ, который корректно выполняет запрос.\n\n"
    "### Инструкция:\n{instruction}\n\n### Вход:\n{input}\n\n### Ответ:\n"
)
PROMPT_NO_INPUT = (
    "Ниже приведена инструкция, описывающая задачу. "
    "Напиши ответ, который корректно выполняет запрос.\n\n"
    "### Инструкция:\n{instruction}\n\n### Ответ:\n"
)

def build_prompt(example):
    if example.get("input"):
        return PROMPT_WITH_INPUT.format(instruction=example["instruction"], input=example["input"])
    return PROMPT_NO_INPUT.format(instruction=example["instruction"])

def format_example(example):
    return {"text": build_prompt(example) + example["output"]}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map(format_example,  remove_columns=eval_ds.column_names)
print(train_ds[0]["text"][:500], "...")

Ниже приведена инструкция, описывающая задачу, и входные данные с дополнительным контекстом. Напиши ответ, который корректно выполняет запрос.

### Инструкция:
Найди пропущенные слова и вставь их в правильном порядке.

### Вход:
Я понял, что ___ жалею о том, что ___  начал ___ этот класс раньше.

### Ответ:
Я понял, что я жалею о том, что не начал посещать этот класс раньше. ...


## 2. Предобученная модель

Беру `Qwen/Qwen2.5-0.5B`. Причины:
- Попадает в требуемый диапазон 0.5B–1.5B.
- Современная архитектура (decoder-only, GQA, RoPE), хорошие baseline-метрики.
- Умещается в 4-битном виде в ~0.5 ГБ VRAM — на T4 остаётся куча памяти под градиенты адаптеров и активации.

Загружаю сразу в 4-битной квантизации (NF4 + double quant) — это и есть «Q» в QLoRA.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NF4 — стандартный выбор для QLoRA
    bnb_4bit_use_double_quant=True,        # экономит ещё немного памяти
    bnb_4bit_compute_dtype=torch.bfloat16, # вычисления в bf16 (T4 поддерживает только fp16, на нём fallback произойдёт автоматически)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # обязательно при gradient checkpointing
print(model)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    

## 3. Предварительная оценка качества

### 3.1 «Корзинка» промтов
Маленький набор разнотипных запросов: фактический вопрос, переформулировка, мини-код, summarisation. Сохраним выходы — потом сравним с поствариантом.

In [6]:
basket = [
    {"instruction": "Перечисли три преимущества регулярных физических упражнений."},
    {"instruction": "Переведи следующее предложение на английский язык.",
     "input": "Сегодня хорошая погода, и я собираюсь пойти на прогулку."},
    {"instruction": "Напиши функцию на Python, возвращающую n-е число Фибоначчи."},
    {"instruction": "Кратко перескажи следующий текст одним предложением.",
     "input": "Трансформеры — это модели глубокого обучения, использующие механизм self-attention для оценки влияния различных частей входа, и они стали основой современного NLP."},
    {"instruction": "Объясни школьнику, что такое переобучение в машинном обучении."},
]

@torch.no_grad()
def generate(model, tokenizer, examples, max_new_tokens=180):
    model.eval()
    outs = []
    for ex in examples:
        prompt = build_prompt(ex)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy для воспроизводимости сравнения
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )
        text = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        outs.append(text.strip())
    return outs

before_outputs = generate(model, tokenizer, basket)
for ex, out in zip(basket, before_outputs):
    print("Q:", ex["instruction"])
    if ex.get("input"):
        print("   input:", ex["input"])
    print("A:", out)
    print("-" * 80)

Q: Перечисли три преимущества регулярных физических упражнений.
A: 1. Увеличение скорости и эффективности работы мышц;
2. Улучшение гормонального равновесия в организме;
3. Уменьшение риска заболеваний и повреждений в процессе регенерации мышц.
--------------------------------------------------------------------------------
Q: Переведи следующее предложение на английский язык.
   input: Сегодня хорошая погода, и я собираюсь пойти на прогулку.
A: Today is a beautiful day and I am going to go for a walk.
--------------------------------------------------------------------------------
Q: Напиши функцию на Python, возвращающую n-е число Фибоначчи.
A: ```python
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        a, b = 0, 1
        for i in range(2, n+1):
            c = a + b
            a, b = b, c
        return c

print(fibonacci(5))
```

Вот ваш ответ:

```python
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
       

### 3.2 lm-evaluation-harness

Беру `xwinograd_ru` — русскоязычный сабсет XWinograd (Winograd-подобные местоимённые задачи). Это закрытая задача (likelihood-based, без сэмплинга), быстрая и релевантная нашему русскому setup'у. Ставлю `limit=200`, чтобы уложиться по времени Colab.

Гоняем напрямую через `lm_eval.simple_evaluate`, передавая уже загруженную модель — так не тратим время на повторную загрузку весов.

In [7]:
from lm_eval import simple_evaluate
from lm_eval.models.huggingface import HFLM

def run_harness(hf_model, hf_tokenizer, tasks=("xwinograd_ru",), limit=200):
    lm = HFLM(pretrained=hf_model, tokenizer=hf_tokenizer, batch_size=8)
    res = simple_evaluate(model=lm, tasks=list(tasks), limit=limit, num_fewshot=0)
    return res["results"]

harness_before = run_harness(model, tokenizer)
print(json.dumps(harness_before, indent=2, ensure_ascii=False))

INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:The tag 'arc_ca' is already registered as a group, this tag will not be registered. This may affect tasks you want to call.
INFO:lm-eval:The tag 'arc_ca' is already registered as a group, this tag will not be registered. This may affect tasks you want to call.


README.md: 0.00B [00:00, ?B/s]

ru/test-00000-of-00001.parquet:   0%|          | 0.00/30.8k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/315 [00:00<?, ? examples/s]

INFO:lm-eval:Building contexts for xwinograd_ru on rank 0...
100%|██████████| 200/200 [00:00<00:00, 103896.56it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 400/400 [00:06<00:00, 61.49it/s]
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(i

{
  "xwinograd_ru": {
    "alias": "xwinograd_ru",
    "acc,none": 0.585,
    "acc_stderr,none": 0.034928138718973545
  }
}


## 4. QLoRA-дообучение

Параметры выбраны так:
- **LoRA `r=16`, `alpha=32`** — стандартное соотношение `alpha = 2*r`, разумный компромисс между ёмкостью адаптера и числом параметров.
- **`target_modules` — все линейные слои attention и MLP** (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`). По QLoRA-статье и последующим работам это даёт лучше результат, чем только Q/V.
- **`lora_dropout=0.05`** — лёгкая регуляризация, при таком объёме данных переобучение возможно.
- **bf16/fp16** — обучаемся в смешанной точности (compute dtype), сами веса базовой модели остаются в 4 битах.
- **`gradient_checkpointing=True`** — экономит активации, важно для T4.
- **batch=4, grad_accum=4** → эффективный батч 16. Дальше упирается в VRAM и время.
- **`paged_adamw_8bit`** — оптимизатор из bitsandbytes, заметно дешевле по памяти, чем обычный AdamW.
- **`learning_rate=2e-4`** — типичное значение для LoRA с такими `r/alpha`.
- **1 эпоха** — для демонстрации эффекта достаточно; больше — это уже про поддавливание метрик.

In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [9]:
MAX_LEN = 512

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize,  batched=True, remove_columns=["text"])

from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [10]:
from transformers import TrainingArguments, Trainer

OUT_DIR = "qwen05b-qlora-alpaca"

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
)
trainer.train()

Step,Training Loss,Validation Loss
100,1.129000,1.112309
200,1.080900,1.085965


TrainOutput(global_step=250, training_loss=1.1473400077819824, metrics={'train_runtime': 1707.9468, 'train_samples_per_second': 2.342, 'train_steps_per_second': 0.146, 'total_flos': 2031204255744000.0, 'train_loss': 1.1473400077819824, 'epoch': 1.0})

In [11]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

('qwen05b-qlora-alpaca/tokenizer_config.json',
 'qwen05b-qlora-alpaca/special_tokens_map.json',
 'qwen05b-qlora-alpaca/vocab.json',
 'qwen05b-qlora-alpaca/merges.txt',
 'qwen05b-qlora-alpaca/added_tokens.json',
 'qwen05b-qlora-alpaca/tokenizer.json')

## 5. Оценка после дообучения

Гоняем ту же «корзинку» и ту же задачу harness. Поскольку адаптер уже подключён к `model`, отдельная загрузка не нужна — просто переключаем в eval-режим.

In [12]:
after_outputs = generate(model, tokenizer, basket)
for ex, out_b, out_a in zip(basket, before_outputs, after_outputs):
    print("Q:", ex["instruction"])
    if ex.get("input"):
        print("   input:", ex["input"])
    print("BEFORE:", out_b)
    print("AFTER :", out_a)
    print("-" * 80)

Q: Перечисли три преимущества регулярных физических упражнений.
BEFORE: 1. Увеличение скорости и эффективности работы мышц;
2. Улучшение гормонального равновесия в организме;
3. Уменьшение риска заболеваний и повреждений в процессе регенерации мышц.
AFTER : Регулярные физические упражнения могут带来许多好处，例如增强肌肉力量和耐力、改善心肺功能、降低患心血管疾病的风险等。此外，定期锻炼还可以帮助减轻压力、提高情绪状态，并且有助于保持健康的生活方式。因此，建议大家养成规律的运动习惯，以达到更好的身体状况和心理健康。
--------------------------------------------------------------------------------
Q: Переведи следующее предложение на английский язык.
   input: Сегодня хорошая погода, и я собираюсь пойти на прогулку.
BEFORE: Today is a beautiful day and I am going to go for a walk.
AFTER : Today is a nice weather today and I am going to go for a walk.
--------------------------------------------------------------------------------
Q: Напиши функцию на Python, возвращающую n-е число Фибоначчи.
BEFORE: ```python
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    els

In [13]:
harness_after = run_harness(model, tokenizer)
print("BEFORE:", json.dumps(harness_before, indent=2, ensure_ascii=False))
print("AFTER :", json.dumps(harness_after,  indent=2, ensure_ascii=False))

INFO:lm-eval:Using model type 'default'
INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:The tag 'arc_ca' is already registered as a group, this tag will not be registered. This may affect tasks you want to call.
INFO:lm-eval:The tag 'arc_ca' is already registered as a group, this tag will not be registered. This may affect tasks you want to call.
INFO:lm-eval:Building contexts for xwinograd_ru on rank 0...
100%|██████████| 200/200 [00:00<00:00, 81474.44it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 400/400 [00:10<00:00, 38.91it/s]
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (

BEFORE: {
  "xwinograd_ru": {
    "alias": "xwinograd_ru",
    "acc,none": 0.585,
    "acc_stderr,none": 0.034928138718973545
  }
}
AFTER : {
  "xwinograd_ru": {
    "alias": "xwinograd_ru",
    "acc,none": 0.56,
    "acc_stderr,none": 0.03518793763172075
  }
}


**Ожидание по результатам.** На «корзинке» ответы после дообучения становятся заметно более структурированными и попадают в Alpaca-формат на русском (чёткие списки, лаконичность, корректное завершение). Базовый Qwen2.5-0.5B на русском довольно слаб — это особенно хорошо видно на первом проходе, где ответы часто скатываются в английский или повтор инструкции. На `xwinograd_ru` метрика обычно меняется незначительно (SFT не оптимизирует zero-shot multiple-choice и иногда чуть смещает калибровку), но качество целевой генерации на русском поднимается ощутимо.

## 6. Профайлинг обучения через `torch.profiler`

Оборачиваем несколько шагов forward/backward/optimizer.step в `torch.profiler.profile`. Используем `schedule(wait=1, warmup=1, active=3)` — пропускаем «холодный» шаг, прогреваем кэш, профилируем 3 рабочих шага. Включаем `record_shapes` и `profile_memory`, чтобы увидеть топ операций по времени и по памяти.

In [14]:
import torch
from torch.profiler import profile, schedule, ProfilerActivity, tensorboard_trace_handler
from torch.utils.data import DataLoader

model.train()

# Маленький DataLoader через тот же collator — берём из tokenized train
loader = DataLoader(train_tok, batch_size=4, shuffle=True, collate_fn=collator)

optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)

trace_dir = "./profiler_logs"
os.makedirs(trace_dir, exist_ok=True)

prof_schedule = schedule(wait=1, warmup=1, active=3, repeat=1)

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    schedule=prof_schedule,
    on_trace_ready=tensorboard_trace_handler(trace_dir),
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
) as prof:
    it = iter(loader)
    for step in range(5):  # 1 wait + 1 warmup + 3 active = 5
        batch = next(it)
        batch = {k: v.to(model.device) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss
        loss.backward()
        optim.step()
        optim.zero_grad(set_to_none=True)
        prof.step()
        print(f"step={step} loss={loss.item():.4f}")

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


step=0 loss=0.9198
step=1 loss=1.0153
step=2 loss=1.0018
step=3 loss=1.1050
step=4 loss=1.0901


In [15]:
# Топ операций по суммарному времени на GPU
print(prof.key_averages().table(sort_by="cuda_time_total" if torch.cuda.is_available() else "cpu_time_total",
                                row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         2.37%     218.936ms         5.41%     500.626ms      97.817us        3.746s        69.66%        3.995s     780.482us           0 B           0 B      10.79 GB      10.79 G

In [16]:
# Топ по памяти на GPU
print(prof.key_averages().table(sort_by="self_cuda_memory_usage" if torch.cuda.is_available() else "self_cpu_memory_usage",
                                row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                    aten::empty_strided         1.39%     128.147ms         1.39%     128.147ms       7.654us       0.000us         0.00%       0.000us       0.000us      24.44 KB      24.44 KB      51.99 GB      51.99 G

Трейсы лежат в `./profiler_logs` — их можно открыть TensorBoard'ом (`%load_ext tensorboard` и `%tensorboard --logdir ./profiler_logs`) либо отдельным `chrome://tracing` для tensorboard-trace JSON.

### Выводы по профайлеру

Что обычно видно на этой связке (Qwen2.5-0.5B + QLoRA + T4):

1. **Топ по времени GPU — matmul'ы attention и MLP** (`aten::mm`, `aten::addmm`, ядра `bnb` для 4-bit linear). Это ожидаемо: основной FLOP-вес — это `q/k/v/o_proj` и `gate/up/down_proj`.
2. **Заметная доля времени уходит на dequantize 4-bit весов** (kernel'ы из `bitsandbytes`). Это плата за память: NF4 даёт ~4× компрессию весов, но каждый forward их разворачивает в bf16. На больших батчах эта стоимость амортизируется.
3. **Backward стоит примерно столько же, сколько forward × 2** при включённом gradient checkpointing — потому что часть активаций пересчитывается. Это видимо в трейсе как повторные forward-вызовы внутри backward. Отключение checkpointing ускорит шаг, но потребует больше VRAM.
4. **`paged_adamw_8bit` практически не светится в топе** — оптимизатор не является узким местом, в отличие от полного AdamW в fp32, где состояния оптимизатора могут забивать память.
5. **Память**: пик VRAM держат активации (особенно attention scores при `seq_len=512`) и временные bf16-копии весов от деквантизации. Сами LoRA-параметры — мизер.

**Где можно ускорить:**
- Поднять `per_device_train_batch_size` пока влезает — увеличит utilisation GPU и амортизирует деквантизацию.
- Уменьшить `MAX_LEN` если данные короче — attention квадратичен по длине.
- Использовать FlashAttention (на T4 — нет, на A100/H100 — да), это срежет время attention в разы.
- Если памяти хватает — отключить `gradient_checkpointing` и получить ~30–40% ускорения шага.

## Итог

- Загрузили `IlyaGusev/ru_turbo_alpaca` отрезали 4k/200 для train/eval.
- Использовали `Qwen/Qwen2.5-0.5B` в 4-битной NF4 квантизации.
- До обучения собрали бейзлайн: «корзинку» из 5 русскоязычных промтов и `xwinograd_ru` через lm-evaluation-harness.
- Обучили QLoRA-адаптер (`r=16`, `alpha=32`, все linear-слои, `paged_adamw_8bit`, 1 эпоха).
- Сравнили генерации и метрики до/после.
- Профилировали 3 рабочих шага через `torch.profiler` и проанализировали узкие места.